# Suite PY — The Python language

Python 3 as a tree language (`@mbse/programs/Python/Syntax`), whose kinds follow Python's own `ast` module: its kinds
and categories, the identifiers it accepts, each kind's own checks, and where each kind and feature exists, which the
versions check.

In [ ]:
import { Syntax as F } from "@mbse/programs/Framework";
import { Python312, Python314, Syntax as S } from "@mbse/programs/Python";
import { PythonStandard } from "@mbse/programs/Python/_Standard";

import { assert, equal } from "./support.js";

const L = S.LANGUAGE;
const version = (minor: number) => new PythonStandard(3, minor);
const n = (spelling: unknown) => new S.Identifier({ spelling } as object);
const v = (spelling: string) => new S.Name({ id: n(spelling) });
const c = (spelling: string) => new S.Constant({ spelling });

## PY-01 · Every kind belongs to the language, the grammar lists them, and their fields are in source order

In [ ]:
const kinds = L.kinds();
equal([...kinds.keys()], S.KINDS.map((k) => k.name));
assert(kinds.size === 86);
equal([...L.categories().keys()], ["Statement", "Expression", "Pattern", "TypeParameter"]);
assert(S.KINDS.every((k) => k.NAME === `Programs.Python.${k.KIND}`));
const grammar = L.grammar() as any;
assert(grammar.language === "Python");
equal(grammar.base, { Python: 300 });
equal(grammar.kinds.find((k: any) => k.kind === "TemplateStr").since, { Python: 314 });
assert(grammar.kinds.find((k: any) => k.kind === "Module").category === "Node");
equal(S.py(12), { Python: 312 });
assert(new S.Comment() instanceof S.Statement);
equal(S.FunctionDef.FIELDS.map((f) => f.name), ["decorator_list", "name", "type_params", "args", "returns", "body"]);
equal(S.Arg.FIELDS.map((f) => f.name), ["arg", "annotation", "default_value"]);
console.log(kinds.size, "kinds");

## PY-02 · Identifiers are Unicode identifiers other than keywords; soft keywords are identifiers

In [ ]:
for (const spelling of ["x_1", "Größe", "_", "match", "case", "type", "lazy", "print"]) equal(L.validate(n(spelling)), []);
equal(L.validate(n("1x")), ["'1x' is not an identifier"]);
equal(L.validate(n("a-b")), ["'a-b' is not an identifier"]);
equal(L.validate(n("")), ["'' is not an identifier"]);
equal(L.validate(n("class")), ["'class' is a keyword"]);
equal(L.validate(n("None")), ["'None' is a keyword"]);
equal(L.validate(n(3n)), ["Identifier.spelling must be a str, got int"]);
equal(L.validate(new S.Identifier()), ["an Identifier needs a spelling"]);
console.log("ok");

## PY-03 · Each kind's own checks: what its lists must hold, and what goes together

In [ ]:
const body = () => [new S.Pass()];
const comments = () => [new S.Comment({ text: " c" })];
const problems: [F.Node, string][] = [
  [new S.DottedName(), "a DottedName needs a name"],
  [new S.ConcatenatedString({ values: [c("'a'")] }), "a ConcatenatedString needs two strings"],
  [new S.Set(), "a Set needs an element"],
  [new S.ListComp({ elt: v("x") }), "a ListComp needs a for clause"],
  [new S.SetComp({ elt: v("x") }), "a SetComp needs a for clause"],
  [new S.GeneratorExp({ elt: v("x") }), "a GeneratorExp needs a for clause"],
  [new S.DictComp({ key: v("x") }), "a DictComp needs a for clause"],
  [new S.Compare({ left: v("a") }), "a Compare needs a comparison"],
  [new S.BoolOp({ op: "and", values: [v("a")] }), "a BoolOp needs two values"],
  [new S.Lambda({ args: new S.Arguments({ args: [new S.Arg({ arg: n("a"), annotation: v("int") })] }), body: v("a") }),
    "a Lambda's parameters have no annotations"],
  [new S.Assign({ value: v("a") }), "an Assign needs a target"],
  [new S.Delete(), "a Delete needs a target"],
  [new S.Import(), "an Import needs a name"],
  [new S.ImportFrom({ module: new S.DottedName({ names: [n("m")] }) }), "an ImportFrom needs a name"],
  [new S.ImportFrom({ names: [new S.Alias()] }), "an ImportFrom needs a module or a level"],
  [new S.Global(), "a Global needs a name"],
  [new S.Raise({ cause: v("c") }), "a Raise with a cause needs an exc"],
  [new S.Nonlocal(), "a Nonlocal needs a name"],
  [new S.If({ test: v("a"), body: comments() }), "an If needs a statement in its body"],
  [new S.If({ test: v("a"), body: body(), orelse: comments() }), "an If needs a statement in its orelse"],
  [new S.While({ test: v("a") }), "a While needs a statement in its body"],
  [new S.For({ target: v("a"), iter: v("b") }), "a For needs a statement in its body"],
  [new S.AsyncFor({ target: v("a"), iter: v("b") }), "an AsyncFor needs a statement in its body"],
  [new S.ExceptHandler(), "an ExceptHandler needs a statement in its body"],
  [new S.Try({ body: body() }), "a Try needs a handler or a finalbody"],
  [new S.Try({ body: body(), orelse: body(), finalbody: body() }), "a Try with an orelse needs a handler"],
  [new S.TryStar({ body: body() }), "a TryStar needs a handler or a finalbody"],
  [new S.With({ body: body() }), "a With needs an item"],
  [new S.AsyncWith({ items: [new S.WithItem({ context_expr: v("a") })] }), "an AsyncWith needs a statement in its body"],
  [new S.MatchCase({ pattern: new S.MatchAs() }), "a MatchCase needs a statement in its body"],
  [new S.Match({ subject: v("a") }), "a Match needs a case"],
  [new S.FunctionDef({ name: n("f"), args: new S.Arguments() }), "a FunctionDef needs a statement in its body"],
  [new S.AsyncFunctionDef({ name: n("f"), args: new S.Arguments() }), "an AsyncFunctionDef needs a statement in its body"],
  [new S.ClassDef({ name: n("C") }), "a ClassDef needs a statement in its body"],
  [new S.MatchMapping({ keys: [c("1")] }), "a MatchMapping needs a pattern for each key"],
  [new S.MatchClass({ cls: v("C"), kwd_attrs: [n("a")] }), "a MatchClass needs a pattern for each keyword"],
  [new S.MatchOr({ patterns: [new S.MatchAs()] }), "a MatchOr needs two patterns"],
  [new S.JoinedStr({ prefix: "fb", quote: "'" }), "JoinedStr.prefix cannot be 'fb'"],
  [new S.TemplateStr({ prefix: "tf", quote: "'" }), "TemplateStr.prefix cannot be 'tf'"],
];
for (const [node, problem] of problems) equal(L.validate(node), [problem]);
equal(L.validate(new S.JoinedStr({ prefix: 1n, quote: "'" } as object)), ["JoinedStr.prefix must be a str, got int"]);
equal(L.validate(new S.Lambda({ args: null, body: v("a") })), ["a Lambda needs an args"]);
for (const prefix of ["f", "F", "rf", "fR", "Rf"]) equal(L.validate(new S.JoinedStr({ prefix, quote: "\"" })), []);
equal(L.validate(new S.TemplateStr({ prefix: "Tr", quote: "\"\"\"" })), []);
const fine = [
  new S.Try({ body: body(), handlers: [new S.ExceptHandler({ body: body() })], orelse: body() }),
  new S.Try({ body: body(), finalbody: [...comments(), ...body()] }),
  new S.ImportFrom({ level: 1n, names: [new S.Alias()] }), new S.BoolOp({ op: "or", values: [v("a"), v("b")] }),
  new S.Lambda({ args: new S.Arguments({ vararg: new S.Arg({ arg: n("a") }) }), body: v("a") }),
];
for (const node of fine) equal(L.validate(node), []);
console.log(problems.length, "problems");

## PY-04 · Where kinds and features exist: since which version of Python

In [ ]:
/** `node` is fine in Python 3.`minor` and has `problem` in the version before. */
function needs(node: F.Node, minor: number, problem: string): void {
  equal(version(minor).check(node), []);
  equal(version(minor - 1).check(node), [problem]);
}

assert(Python314.STANDARD.name() === "Python 3.14" && String(Python312.STANDARD) === "<standard Python 3.12>");
assert(Python314.STANDARD.version === 314 && version(0).name() === "Python 3.0");
const generator = new S.Comprehension({ target: v("x"), iter: v("y") });
needs(new S.Await({ value: v("a") }), 5, "Await needs Python 3.5");
needs(new S.BinOp({ left: v("a"), op: "@", right: v("b") }), 5, "BinOp.op '@' needs Python 3.5");
needs(new S.AugAssign({ target: v("a"), op: "@", value: v("b") }), 5, "AugAssign.op '@' needs Python 3.5");
needs(new S.AnnAssign({ target: v("a"), annotation: v("int") }), 6, "AnnAssign needs Python 3.6");
needs(new S.JoinedStr({ prefix: "f", quote: "'" }), 6, "JoinedStr needs Python 3.6");
needs(new S.Comprehension({ is_async: true, target: v("x"), iter: v("y") }), 6, "Comprehension.is_async needs Python 3.6");
needs(new S.NamedExpr({ target: v("a"), value: v("b") }), 8, "NamedExpr needs Python 3.8");
needs(new S.Arguments({ posonlyargs: [new S.Arg({ arg: n("a") })] }), 8, "Arguments.posonlyargs needs Python 3.8");
needs(new S.FormattedValue({ value: v("a"), debug: true }), 8, "FormattedValue.debug needs Python 3.8");
const decorated = new S.FunctionDef({ decorator_list: [new S.Subscript({ value: v("d"), slice: c("0") })], name: n("f"),
  args: new S.Arguments(), body: body() });
needs(decorated, 9, "FunctionDef with a decorator that is not a dotted name needs Python 3.9");
const dotted = new S.Call({ func: new S.Attribute({ value: v("d"), attr: n("e") }) });
equal(version(8).check(new S.ClassDef({ decorator_list: [v("d"), dotted], name: n("C"), body: body() })), []);
needs(new S.MatchSingleton({ value: "None" }), 10, "MatchSingleton needs Python 3.10");
needs(new S.Match({ subject: v("a") }), 10, "Match needs Python 3.10");
needs(new S.TryStar(), 11, "TryStar needs Python 3.11");
needs(new S.Subscript({ value: v("a"), slice: new S.Starred({ value: v("b") }) }), 11,
  "Subscript with an unpacked index needs Python 3.11");
needs(new S.Subscript({ value: v("a"), slice: new S.Tuple({ elts: [v("i"), new S.Starred({ value: v("b") })] }) }), 11,
  "Subscript with an unpacked index needs Python 3.11");
equal(version(10).check(new S.Subscript({ value: v("a"), slice: new S.Tuple({ elts: [v("i")] }) })), []);
needs(new S.TypeVar({ name: n("T") }), 12, "TypeVar needs Python 3.12");
const generic = new S.ClassDef({ name: n("C"), type_params: [new S.TypeVar({ name: n("T") })], body: body() });
equal(version(11).check(generic), ["ClassDef.type_params needs Python 3.12", "type_params[0]: TypeVar needs Python 3.12"]);
needs(new S.ParamSpec({ name: n("P"), default_value: v("a") }), 13, "ParamSpec.default_value needs Python 3.13");
needs(new S.TypeVarTuple({ name: n("Ts"), default_value: v("a") }), 13, "TypeVarTuple.default_value needs Python 3.13");
needs(new S.TemplateStr({ prefix: "t", quote: "'" }), 14, "TemplateStr needs Python 3.14");
needs(new S.ExceptHandler({ type: new S.Tuple({ elts: [v("A"), v("B")] }), body: body() }), 14,
  "ExceptHandler with unparenthesized types needs Python 3.14");
equal(version(13).check(new S.ExceptHandler({
  type: new S.Parenthesized({ value: new S.Tuple({ elts: [v("A")] }) }), body: body() })), []);
needs(new S.Import({ is_lazy: true, names: [new S.Alias({ name: new S.DottedName({ names: [n("m")] }) })] }), 15,
  "Import.is_lazy needs Python 3.15");
needs(new S.ImportFrom({ is_lazy: true, level: 1n, names: [new S.Alias()] }), 15, "ImportFrom.is_lazy needs Python 3.15");
needs(new S.ListComp({ elt: new S.Starred({ value: v("x") }), generators: [generator] }), 15,
  "ListComp of unpacked elements needs Python 3.15");
needs(new S.GeneratorExp({ elt: new S.Starred({ value: v("x") }), generators: [generator] }), 15,
  "GeneratorExp of unpacked elements needs Python 3.15");
needs(new S.SetComp({ elt: new S.Starred({ value: v("x") }), generators: [generator] }), 15,
  "SetComp of unpacked elements needs Python 3.15");
needs(new S.DictComp({ key: v("x"), generators: [generator] }), 15, "DictComp of unpacked mappings needs Python 3.15");
equal(version(14).check(new S.DictComp({ key: v("x"), value: v("y"), generators: [generator] })), []);
equal(Python312.check(new S.Module({ body: [new S.Expr({ value: new S.TemplateStr({ prefix: "t", quote: "'" }) })] })),
  ["body[0].value: TemplateStr needs Python 3.14"]);
console.log("ok");